# Week 2 Day 2 assignment, stretch: retrieval in a sub-agent

A research sub-agent gets `qdrant-find` and task 2's customer note tools, and the UnderCover agent calls it as a tool. Run tasks 1 and 2 first; their runs are the comparison.

What makes it hard is not the code:

- **The scoring breaks silently.** `decide` reads the searches and the model cost from the UnderCover agent's messages, and the sub-agent's happen in a conversation of its own. So the tool returns its reply as content and the raw search results and cost as the tool message's artifact, which the model never sees and `decide` adds in.
- **The sub-agent sees only the request.** It never sees the claim or the policy date, and the storm and strike claims turn on that date. So it quotes facts word for word, and `relayed_fact` checks that the needed fact reached the UnderCover agent, not only that a search found it.
- **Async and MCP.** The Qdrant tools only run async, so the tool is an `async` function, and both agents keep `ToolRetryMiddleware` for Qdrant's one-process lock. The sub-agent's calls nest in the UnderCover agent's Langfuse trace with no extra code.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from week2day2_common import (
    add_customer_note, calculate_payout, compare, decide, load_runs, luna, make_system_prompt, memory_search_tools,
    read_customer_notes, record_decision, reset_notes, run_experiment,
)

In [ ]:
RESEARCHER_INSTRUCTIONS = """You are the research assistant to a claims handler at UnderCover Travel Insurance. You look things up and keep the customer notes; you never decide claims.

You have two memories:
- qdrant-find searches UnderCover's knowledge base: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators.
- read_customer_notes and add_customer_note read and add to UnderCover's notes on each customer.

When you are asked about products, events, places or a customer, search for each one separately. Reply with every fact you find that bears on the request, quoted word for word with its dates and figures, because the handler decides on details you can't see. If you find nothing on something, say so.
When you are asked to add a note on a customer, add it as you are given it and confirm.
"""

memory_search = await memory_search_tools()
researcher = create_agent(model=luna, tools=[*memory_search, read_customer_notes, add_customer_note], system_prompt=RESEARCHER_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])


@tool(parse_docstring=True, response_format="content_and_artifact")
async def ask_researcher(request: str) -> tuple[str, dict]:
    """Ask your research assistant to look things up in UnderCover's knowledge base and customer notes, or to add a note on a customer.

    The assistant can't see the claim, so name each product, event, place and customer id, and say what you need to know.

    Args:
        request: What to look up, or the note to add and the customer id, in full
    """
    result = await researcher.ainvoke({"messages": [{"role": "user", "content": request}]})
    messages = result["messages"]
    retrieved = "\n".join(message.text for message in messages if message.type == "tool" and message.name == "qdrant-find")
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in messages if message.type == "ai")
    return messages[-1].text, {"retrieved": retrieved, "cost": cost}

In [ ]:
RESEARCH_LINE = "Before you decide, ask your research assistant about any product, event or place in the claim that you need to know more about, and for the customer's notes. After you record your decision, ask it to add a short note on the customer for the next handler: what was claimed, what you decided and anything still outstanding.\n"

research_agent = create_agent(model=luna, tools=[record_decision, calculate_payout, ask_researcher], system_prompt=make_system_prompt(RESEARCH_LINE), middleware=[ToolRetryMiddleware(max_retries=3)])


async def research_task(*, item, **kwargs):
    return await decide(research_agent, item.input, search_tools=[])


research_table = run_experiment("Payout tool and research sub-agent", research_task, "researcher", before_run=reset_notes)
tables = {
    "Memory (task 1)": load_runs("qdrant"),
    "Word lookup (task 1)": load_runs("word_lookup"),
    "Memory and notes (task 2)": load_runs("notes"),
    "Research sub-agent": research_table,
}
print(f"The sub-agent's share of the model cost: {research_table['sub_agent_cost'].sum() / research_table['llm_cost'].sum():.0%}")
compare(tables)

In [ ]:
research_table[~research_table["right"]][["claim_id", "run", "handler", "agent", "found_fact", "relayed_fact", "rationale"]]

## Conclusion

**The sub-agent works, but on this task it costs more, takes longer and refers more claims, with nothing gained in return.**

| | Memory (task 1) | Memory and notes (task 2) | Research sub-agent |
|---|---:|---:|---:|
| Right, over 3 runs | 70 of 75 | 74 of 75 | 64 of 75 (21, 23, 20) |
| Straight-through | 89% | 91% | 80% |
| Cost of mistakes, per run | $590 | $17 | $218 |
| Model cost, per run | $0.012 | $0.016 | $0.030 |
| Seconds per claim | 14.7 | 14.1 | 31.1 |

- The facts got through. The searches found the needed fact every time, and the sub-agent quoted it word for word. `relayed_fact` is 98% only because one reply quoted the Meridian Air press release but called it "a press release dated August 5"; that claim was right. The UnderCover agent learned to pass on the policy date, so the sub-agent could check the timing itself.
- The cost is the extra agent. The UnderCover agent called the sub-agent twice per claim, once to look things up and once to add the note. Each call is a full agent loop with its own prompt, so the sub-agent took 48% of the model cost, and the claims took twice as long.
- The mistakes are needless referrals: 9 of the 11. With the facts in hand, the agent referred claims the wording settles: "the research result describing the Lantern Run as grade 2 is not dated" (CLM-1317), and a storm that "may exclude the loss" (CLM-1305). Facts reported back by an assistant seem to carry less weight than facts it found itself. The cost of mistakes is lower than task 1's only because a needless referral costs $50 and task 1 had a wrongful decline at $1,500.

A sub-agent earns its place when retrieval would flood the main agent's context: many searches, long documents, or a separate model for research. Here each search returns 10 short facts, so the agent is better off searching for itself.